# Load Data

In [2]:
import json
import networkx as nx
from glob import glob
from gurobipy import *

dataset_1 = []
dataset_2 = []

for f_name in glob('dataset1/*.json'):
    with open(f_name) as f:
        dataset_1.append(json.load(f))

for f_name in glob('dataset2/*.json'):
    with open(f_name) as f:
        dataset_2.append(json.load(f))

# Solve Natural Formulation Function

In [3]:
def solveNaturalFormulation(data, time_limit):
    m = data['m']
    n = data['n']
    R = data['R']
    r = data['r']
    edges = data['edges']
    G = nx.Graph(edges)

    myModel = Model('finalProject')

    myModel.setParam('TimeLimit', time_limit)
    myModel.setParam('OutputFlag', 0)

    x = [[0 for j in range(n)] for i in range(m)]
    y = [0 for j in range(n)]

    for i in range(m):
        for j in range(n):
            curVar = myModel.addVar(vtype=GRB.BINARY, name=f"x{i+1},{j+1}")
            x[i][j] = curVar

    for j in range(n):
        curVar = myModel.addVar(vtype=GRB.BINARY, name=f"y{j+1}")
        y[j] = curVar

    myModel.update()

    objExpr = LinExpr()
    for j in range(n):
        objExpr += y[j]

    myModel.setObjective(objExpr, GRB.MINIMIZE)

    for i in range(m):
        constExpr = LinExpr()
        for j in range(n):
            curVar = x[i][j]

            constExpr += curVar
        myModel.addLConstr(lhs=constExpr, sense=GRB.EQUAL, rhs=1, name=f"exam{i+1}")

    for j in range(n):
        lhsExpr = LinExpr()
        rhsExpr = LinExpr()
        for i in range(m):
            curVar = x[i][j]

            lhsExpr += r[i] * curVar
        rhsExpr += R * y[j]
        myModel.addLConstr(lhs=lhsExpr, sense=GRB.LESS_EQUAL, rhs=rhsExpr, name=f"proctor{j+1}")

    for j in range(n):
        for e in G.edges:
            i = e[0]
            i_prime = e[1]

            constExpr = LinExpr()

            curVar1 = x[i-1][j]
            curVar2 = x[i_prime-1][j]

            constExpr += curVar1 + curVar2

            myModel.addLConstr(lhs=constExpr, sense=GRB.LESS_EQUAL, rhs=1, name=f"conflict{i},{i_prime},{j}")

    myModel.update()

    relaxedModel = myModel.relax()

    relaxedModel.setParam('OutputFlag', 0)

    relaxedModel.optimize()

    myModel.optimize()

    print("\n")
    print("Natural Formulation")
    print(f"Objective value reached: {myModel.ObjVal}")
    print(f"LP relaxation value: {relaxedModel.ObjVal}")
    print(f"Node count: {myModel.NodeCount}")
    print(f"Runtime: {myModel.Runtime}")
    print(f"Solved to Optimality?: {myModel.Status == GRB.OPTIMAL}")
    print("\n")
    

# Solve Clique-Strengthened Formulation Function

In [4]:
def solveCliqueStrengthenedFormulation(data, time_limit):
    m = data['m']
    n = data['n']
    R = data['R']
    r = data['r']
    edges = data['edges']
    G = nx.Graph(edges)
    cliques = []

    for edge in G.edges:
        edge = list(edge)

        edgeClique = sorted(list(nx.find_cliques(G, edge))[0])

        if edgeClique not in cliques:
            cliques.append(edgeClique)

    myModel = Model('finalProject')

    myModel.setParam('TimeLimit', time_limit)
    myModel.setParam('OutputFlag', 0)

    x = [[0 for j in range(n)] for i in range(m)]
    y = [0 for j in range(n)]

    for i in range(m):
        for j in range(n):
            curVar = myModel.addVar(vtype=GRB.BINARY, name=f"x{i+1},{j+1}")
            x[i][j] = curVar

    for j in range(n):
        curVar = myModel.addVar(vtype=GRB.BINARY, name=f"y{j+1}")
        y[j] = curVar

    myModel.update()

    objExpr = LinExpr()
    for j in range(n):
        objExpr += y[j]

    myModel.setObjective(objExpr, GRB.MINIMIZE)

    for i in range(m):
        constExpr = LinExpr()
        for j in range(n):
            curVar = x[i][j]

            constExpr += curVar
        myModel.addLConstr(lhs=constExpr, sense=GRB.EQUAL, rhs=1, name=f"exam{i+1}")

    for j in range(n):
        lhsExpr = LinExpr()
        rhsExpr = LinExpr()
        for i in range(m):
            curVar = x[i][j]

            lhsExpr += r[i] * curVar
        rhsExpr += R * y[j]
        myModel.addLConstr(lhs=lhsExpr, sense=GRB.LESS_EQUAL, rhs=rhsExpr, name=f"proctor{j+1}")

    for j in range(n):
        for clique in cliques:
            constExpr = LinExpr()

            for i in clique:
                curVar = x[i-1][j]

                constExpr += curVar

            myModel.addLConstr(lhs=constExpr, sense=GRB.LESS_EQUAL, rhs=1, name=f"conflict{','.join(str(i) for i in clique)},{j}")

    myModel.update()

    relaxedModel = myModel.relax()

    relaxedModel.setParam('OutputFlag', 0)

    relaxedModel.optimize()

    myModel.optimize()

    print("\n")
    print("Clique-Strengthened Formulation")
    print(f"Objective value reached: {myModel.ObjVal}")
    print(f"LP relaxation value: {relaxedModel.ObjVal}")
    print(f"Node count: {myModel.NodeCount}")
    print(f"Runtime: {myModel.Runtime}")
    print(f"Solved to Optimality?: {myModel.Status == GRB.OPTIMAL}")
    print("\n")

# Solve Extended Formulation Function

In [5]:
def solveExtendedFormulation(data, time_limit):
    m = data['m']
    n = data['n']
    R = data['R']
    r = data['r']
    edges = data['edges']
    G = nx.Graph(edges)
    G_complement = nx.complement(G)
    stableSets = list(nx.enumerate_all_cliques(G_complement))
    S = []

    for s in stableSets:
        proctorRequirement = 0

        for i in s:
            proctorRequirement += r[i-1]
        
        if proctorRequirement <= R:
            S.append(s)

    myModel = Model('finalProject')

    myModel.setParam('TimeLimit', time_limit)
    myModel.setParam('OutputFlag', 0)

    x = [0 for s in range(len(S))]

    for i, s in enumerate(S):
        curVar = myModel.addVar(vtype=GRB.BINARY, name=f"x{','.join(str(i) for i in s)}")
        x[i] = curVar

    myModel.update()

    objExpr = LinExpr()
    for i in range(len(S)):
        objExpr += x[i]

    myModel.setObjective(objExpr, GRB.MINIMIZE)

    for i in range(1, m + 1):
        constExpr = LinExpr()
        for k, s in enumerate(S):
            if i in s:
                curVar = x[k]
                
                constExpr += curVar
        myModel.addLConstr(lhs=constExpr, sense=GRB.EQUAL, rhs=1, name=f"exam{i}")

    myModel.update()

    relaxedModel = myModel.relax()

    relaxedModel.setParam('OutputFlag', 0)

    relaxedModel.optimize()

    myModel.optimize()

    print("\n")
    print("Extended Formulation")
    print(f"Objective value reached: {myModel.ObjVal}")
    print(f"LP relaxation value: {relaxedModel.ObjVal}")
    print(f"Node count: {myModel.NodeCount}")
    print(f"Runtime: {myModel.Runtime}")
    print(f"Solved to Optimality?: {myModel.Status == GRB.OPTIMAL}")
    print("\n")

# Dataset 1

## Instance 1

In [6]:
data = dataset_1[0]

solveNaturalFormulation(data, 120)

solveCliqueStrengthenedFormulation(data, 120)

solveExtendedFormulation(data, 120)

Set parameter Username
Set parameter LicenseID to value 2719877
Academic license - for non-commercial use only - expires 2026-10-08
Set parameter TimeLimit to value 120


Natural Formulation
Objective value reached: 8.0
LP relaxation value: 6.319999999999999
Node count: 21.0
Runtime: 7.259000062942505
Solved to Optimality?: True


Set parameter TimeLimit to value 120


Clique-Strengthened Formulation
Objective value reached: 8.0
LP relaxation value: 6.320000000000001
Node count: 4886.0
Runtime: 19.197999954223633
Solved to Optimality?: True


Set parameter TimeLimit to value 120


Extended Formulation
Objective value reached: 8.0
LP relaxation value: 7.183333333333333
Node count: 1.0
Runtime: 0.039999961853027344
Solved to Optimality?: True




## Instance 2

In [7]:
data = dataset_1[1]

solveNaturalFormulation(data, 120)

solveCliqueStrengthenedFormulation(data, 120)

solveExtendedFormulation(data, 120)

Set parameter TimeLimit to value 120




Natural Formulation
Objective value reached: 9.0
LP relaxation value: 7.319999999999999
Node count: 7026.0
Runtime: 21.272000074386597
Solved to Optimality?: True


Set parameter TimeLimit to value 120


Clique-Strengthened Formulation
Objective value reached: 9.0
LP relaxation value: 7.320000000000003
Node count: 11586.0
Runtime: 39.81500005722046
Solved to Optimality?: True


Set parameter TimeLimit to value 120


Extended Formulation
Objective value reached: 9.0
LP relaxation value: 7.920634920634921
Node count: 1.0
Runtime: 0.0970001220703125
Solved to Optimality?: True




## Instance 3

In [8]:
data = dataset_1[2]

solveNaturalFormulation(data, 120)

solveCliqueStrengthenedFormulation(data, 120)

solveExtendedFormulation(data, 120)

Set parameter TimeLimit to value 120


Natural Formulation
Objective value reached: 9.0
LP relaxation value: 7.880000000000001
Node count: 339.0
Runtime: 14.611000061035156
Solved to Optimality?: True


Set parameter TimeLimit to value 120


Clique-Strengthened Formulation
Objective value reached: 9.0
LP relaxation value: 7.879999999999999
Node count: 6249.0
Runtime: 25.638000011444092
Solved to Optimality?: True


Set parameter TimeLimit to value 120


Extended Formulation
Objective value reached: 9.0
LP relaxation value: 8.11871227364185
Node count: 1.0
Runtime: 0.07299995422363281
Solved to Optimality?: True




## Instance 4

In [9]:
data = dataset_1[3]

solveNaturalFormulation(data, 120)

solveCliqueStrengthenedFormulation(data, 120)

solveExtendedFormulation(data, 120)

Set parameter TimeLimit to value 120


Natural Formulation
Objective value reached: 14.0
LP relaxation value: 12.840000000000002
Node count: 14259.0
Runtime: 120.02499985694885
Solved to Optimality?: False


Set parameter TimeLimit to value 120


Clique-Strengthened Formulation
Objective value reached: 14.0
LP relaxation value: 12.84
Node count: 3015.0
Runtime: 120.0680000782013
Solved to Optimality?: False


Set parameter TimeLimit to value 120


Extended Formulation
Objective value reached: 13.0
LP relaxation value: 12.839999999999996
Node count: 1.0
Runtime: 1.680999994277954
Solved to Optimality?: True




## Instance 5

In [10]:
data = dataset_1[4]

solveNaturalFormulation(data, 120)

solveCliqueStrengthenedFormulation(data, 120)

solveExtendedFormulation(data, 120)

Set parameter TimeLimit to value 120


Natural Formulation
Objective value reached: 13.0
LP relaxation value: 12.400000000000004
Node count: 1.0
Runtime: 17.486999988555908
Solved to Optimality?: True


Set parameter TimeLimit to value 120


Clique-Strengthened Formulation
Objective value reached: 13.0
LP relaxation value: 12.400000000000002
Node count: 282.0
Runtime: 28.002000093460083
Solved to Optimality?: True


Set parameter TimeLimit to value 120


Extended Formulation
Objective value reached: 13.0
LP relaxation value: 12.400000000000002
Node count: 1.0
Runtime: 0.312000036239624
Solved to Optimality?: True




## Instance 6

In [11]:
data = dataset_1[5]

solveNaturalFormulation(data, 120)

solveCliqueStrengthenedFormulation(data, 120)

solveExtendedFormulation(data, 120)

Set parameter TimeLimit to value 120


Natural Formulation
Objective value reached: 14.0
LP relaxation value: 12.88
Node count: 2401.0
Runtime: 120.02300000190735
Solved to Optimality?: False


Set parameter TimeLimit to value 120


Clique-Strengthened Formulation
Objective value reached: 14.0
LP relaxation value: 12.88
Node count: 2384.0
Runtime: 120.11800003051758
Solved to Optimality?: False


Set parameter TimeLimit to value 120


Extended Formulation
Objective value reached: 13.0
LP relaxation value: 12.88
Node count: 343.0
Runtime: 2.503000020980835
Solved to Optimality?: True




## Instance 7

In [12]:
data = dataset_1[6]

solveNaturalFormulation(data, 120)

solveCliqueStrengthenedFormulation(data, 120)

solveExtendedFormulation(data, 120)

Set parameter TimeLimit to value 120


Natural Formulation
Objective value reached: 13.0
LP relaxation value: 12.679999999999998
Node count: 5153.0
Runtime: 114.46099996566772
Solved to Optimality?: True


Set parameter TimeLimit to value 120


Clique-Strengthened Formulation
Objective value reached: 14.0
LP relaxation value: 12.680000000000007
Node count: 2203.0
Runtime: 120.1710000038147
Solved to Optimality?: False


Set parameter TimeLimit to value 120


Extended Formulation
Objective value reached: 13.0
LP relaxation value: 12.679999999999998
Node count: 1.0
Runtime: 2.683000087738037
Solved to Optimality?: True




## Instance 8

In [13]:
data = dataset_1[7]

solveNaturalFormulation(data, 120)

solveCliqueStrengthenedFormulation(data, 120)

solveExtendedFormulation(data, 120)

Set parameter TimeLimit to value 120


Natural Formulation
Objective value reached: 16.0
LP relaxation value: 15.36
Node count: 1.0
Runtime: 3.0959999561309814
Solved to Optimality?: True


Set parameter TimeLimit to value 120


Clique-Strengthened Formulation
Objective value reached: 16.0
LP relaxation value: 15.359999999999998
Node count: 2312.0
Runtime: 102.22200012207031
Solved to Optimality?: True


Set parameter TimeLimit to value 120


Extended Formulation
Objective value reached: 16.0
LP relaxation value: 15.36
Node count: 1.0
Runtime: 0.5240001678466797
Solved to Optimality?: True




## Instance 9

In [14]:
data = dataset_1[8]

solveNaturalFormulation(data, 120)

solveCliqueStrengthenedFormulation(data, 120)

solveExtendedFormulation(data, 120)

Set parameter TimeLimit to value 120


Natural Formulation
Objective value reached: 15.0
LP relaxation value: 14.160000000000002
Node count: 1.0
Runtime: 6.5350000858306885
Solved to Optimality?: True


Set parameter TimeLimit to value 120


Clique-Strengthened Formulation
Objective value reached: 16.0
LP relaxation value: 14.159999999999997
Node count: 1414.0
Runtime: 120.09999990463257
Solved to Optimality?: False


Set parameter TimeLimit to value 120


Extended Formulation
Objective value reached: 15.0
LP relaxation value: 14.160000000000004
Node count: 1.0
Runtime: 0.6979999542236328
Solved to Optimality?: True




## Instance 10

In [15]:
data = dataset_1[9]

solveNaturalFormulation(data, 120)

solveCliqueStrengthenedFormulation(data, 120)

solveExtendedFormulation(data, 120)

Set parameter TimeLimit to value 120


Natural Formulation
Objective value reached: 16.0
LP relaxation value: 14.68
Node count: 3589.0
Runtime: 120.05099987983704
Solved to Optimality?: False


Set parameter TimeLimit to value 120


Clique-Strengthened Formulation
Objective value reached: 16.0
LP relaxation value: 14.680000000000001
Node count: 858.0
Runtime: 120.10899996757507
Solved to Optimality?: False


Set parameter TimeLimit to value 120


Extended Formulation
Objective value reached: 15.0
LP relaxation value: 14.680000000000003
Node count: 1.0
Runtime: 0.6050000190734863
Solved to Optimality?: True




# Dataset 2

## Instance 1

In [16]:
data = dataset_2[0]

solveNaturalFormulation(data, 600)

solveCliqueStrengthenedFormulation(data, 600)

Set parameter TimeLimit to value 600


Natural Formulation
Objective value reached: 17.0
LP relaxation value: 16.32
Node count: 175.0
Runtime: 20.873000144958496
Solved to Optimality?: True


Set parameter TimeLimit to value 600


Clique-Strengthened Formulation
Objective value reached: 17.0
LP relaxation value: 16.32
Node count: 308.0
Runtime: 70.17400002479553
Solved to Optimality?: True




## Instance 2

In [17]:
data = dataset_2[1]

solveNaturalFormulation(data, 600)

solveCliqueStrengthenedFormulation(data, 600)

Set parameter TimeLimit to value 600


Natural Formulation
Objective value reached: 17.0
LP relaxation value: 16.080000000000002
Node count: 1.0
Runtime: 5.0350000858306885
Solved to Optimality?: True


Set parameter TimeLimit to value 600


Clique-Strengthened Formulation
Objective value reached: 17.0
LP relaxation value: 16.08
Node count: 1.0
Runtime: 45.32800006866455
Solved to Optimality?: True




## Instance 3

In [18]:
data = dataset_2[2]

solveNaturalFormulation(data, 600)

solveCliqueStrengthenedFormulation(data, 600)

Set parameter TimeLimit to value 600


Natural Formulation
Objective value reached: 17.0
LP relaxation value: 16.28
Node count: 255.0
Runtime: 37.5770001411438
Solved to Optimality?: True


Set parameter TimeLimit to value 600


Clique-Strengthened Formulation
Objective value reached: 17.0
LP relaxation value: 16.28
Node count: 1.0
Runtime: 37.581000089645386
Solved to Optimality?: True




## Instance 4

In [19]:
data = dataset_2[3]

solveNaturalFormulation(data, 600)

solveCliqueStrengthenedFormulation(data, 600)

Set parameter TimeLimit to value 600


Natural Formulation
Objective value reached: 18.0
LP relaxation value: 16.76
Node count: 16167.0
Runtime: 600.0660002231598
Solved to Optimality?: False


Set parameter TimeLimit to value 600


Clique-Strengthened Formulation
Objective value reached: 18.0
LP relaxation value: 16.75999999999999
Node count: 2229.0
Runtime: 600.2200000286102
Solved to Optimality?: False




## Instance 5

In [20]:
data = dataset_2[4]

solveNaturalFormulation(data, 600)

solveCliqueStrengthenedFormulation(data, 600)

Set parameter TimeLimit to value 600


Natural Formulation
Objective value reached: 17.0
LP relaxation value: 16.599999999999998
Node count: 2410.0
Runtime: 253.29499983787537
Solved to Optimality?: True


Set parameter TimeLimit to value 600


Clique-Strengthened Formulation
Objective value reached: 17.0
LP relaxation value: 16.599999999999994
Node count: 1077.0
Runtime: 212.61800003051758
Solved to Optimality?: True




## Instance 6

In [21]:
data = dataset_2[5]

solveNaturalFormulation(data, 600)

solveCliqueStrengthenedFormulation(data, 600)

Set parameter TimeLimit to value 600


Natural Formulation
Objective value reached: 18.0
LP relaxation value: 17.759999999999998
Node count: 2190.0
Runtime: 134.73200011253357
Solved to Optimality?: True


Set parameter TimeLimit to value 600


Clique-Strengthened Formulation
Objective value reached: 18.0
LP relaxation value: 17.759999999999998
Node count: 1975.0
Runtime: 537.7990000247955
Solved to Optimality?: True




## Instance 7

In [22]:
data = dataset_2[6]

solveNaturalFormulation(data, 600)

solveCliqueStrengthenedFormulation(data, 600)

Set parameter TimeLimit to value 600


Natural Formulation
Objective value reached: 18.0
LP relaxation value: 16.880000000000003
Node count: 18386.0
Runtime: 600.0409998893738
Solved to Optimality?: False


Set parameter TimeLimit to value 600


Clique-Strengthened Formulation
Objective value reached: 18.0
LP relaxation value: 16.879999999999924
Node count: 2122.0
Runtime: 600.1189999580383
Solved to Optimality?: False




## Instance 8

In [23]:
data = dataset_2[7]

solveNaturalFormulation(data, 600)

solveCliqueStrengthenedFormulation(data, 600)

Set parameter TimeLimit to value 600


Natural Formulation
Objective value reached: 17.0
LP relaxation value: 16.839999999999996
Node count: 6053.0
Runtime: 372.52800011634827
Solved to Optimality?: True


Set parameter TimeLimit to value 600


Clique-Strengthened Formulation
Objective value reached: 18.0
LP relaxation value: 16.84
Node count: 2905.0
Runtime: 600.1849999427795
Solved to Optimality?: False




## Instance 9

In [24]:
data = dataset_2[8]

solveNaturalFormulation(data, 600)

solveCliqueStrengthenedFormulation(data, 600)

Set parameter TimeLimit to value 600


Natural Formulation
Objective value reached: 19.0
LP relaxation value: 18.0
Node count: 5715.0
Runtime: 600.1039998531342
Solved to Optimality?: False


Set parameter TimeLimit to value 600


Clique-Strengthened Formulation
Objective value reached: 19.0
LP relaxation value: 18.000000000000004
Node count: 1602.0
Runtime: 600.2309999465942
Solved to Optimality?: False


